# Предсказание стоимости квартир

В этом ноутбуке я читаю данные о квартирах, очищаю их, добавляю несколько новых признаков, обучаю модель CatBoost и сохраняю результат для Telegram-бота.

In [ ]:
# Импортируем библиотеки для таблиц, графиков и обучения модели
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Показываем больше колонок, когда выводим таблицу
pd.set_option('display.max_columns', 100)

Здесь подключаются библиотеки, которые нужны для работы с таблицей, построения графиков и обучения модели.

In [ ]:
# Указываем, где лежат данные и куда сохранять модели
DATA_PATH = Path('input_data.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../input_data.csv')

MODELS_DIR = Path('models')
if not MODELS_DIR.exists():
    MODELS_DIR = Path('../models')

MODELS_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError('Не найден файл input_data.csv')

DATA_PATH


Файл с данными лежит в корне проекта, а готовые модели сохраняются в папку `models`.

In [ ]:
# Датасет большой, поэтому читаем его частями и берем случайную выборку
chunk_size = 250_000
sample_part = 0.08
max_rows = 900_000
random_seed = 42
test_size = 0.2

rng = np.random.default_rng(random_seed)
parts = []

for chunk in pd.read_csv(DATA_PATH, sep=';', chunksize=chunk_size):
    sample_mask = rng.random(len(chunk)) < sample_part
    parts.append(chunk.loc[sample_mask].copy())

    rows_now = sum(len(part) for part in parts)
    print('строк в выборке:', rows_now)

    if rows_now >= max_rows:
        break

df = pd.concat(parts, ignore_index=True)

if len(df) > max_rows:
    df = df.sample(max_rows, random_state=random_seed).reset_index(drop=True)

print('размер таблицы:', df.shape)
df.head()

Файл большой, поэтому я читаю его частями. Так ноутбук не пытается загрузить весь датасет сразу.

In [ ]:
# Проверяем размер, названия колонок и первые строки
print(df.shape)
print(df.columns.tolist())
df.head()

Сначала просто смотрю размер таблицы, названия колонок и первые строки.

In [ ]:
# Считаем пропуски по всем колонкам
missing = pd.DataFrame({
    'column': df.columns,
    'empty_count': df.isna().sum().values,
    'empty_percent': (df.isna().mean().values * 100).round(2),
    'type': df.dtypes.astype(str).values,
})

missing.sort_values('empty_percent', ascending=False)

Так видно, в каких колонках больше всего пропусков. Это помогает понять, какие данные нужно чистить.

In [ ]:
# Приводим нужные поля к числовому типу
num_cols = [
    'price', 'level', 'levels', 'rooms', 'area', 'kitchen_area',
    'geo_lat', 'geo_lon', 'building_type', 'object_type',
    'postal_code', 'id_region',
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Из даты оставляем год и месяц объявления
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year.fillna(2021).astype(int)
df['month'] = df['date'].dt.month.fillna(1).astype(int)

df[['price', 'area', 'rooms', 'level', 'levels', 'kitchen_area']].describe().T

Перед обучением важно привести признаки к правильным типам: числовые поля должны быть числами, а дата разбирается отдельно.

In [ ]:
# Убираем явно некорректные объявления и грубые выбросы
before = len(df)

df = df.dropna(subset=['price', 'area', 'level', 'levels', 'rooms', 'geo_lat', 'geo_lon', 'id_region'])

df = df[df['price'].between(300_000, 80_000_000)]
df = df[df['area'].between(10, 250)]
df = df[df['level'].between(1, 80)]
df = df[df['levels'].between(1, 80)]
df = df[df['level'] <= df['levels']]
df = df[df['rooms'].between(-1, 6)]
df = df[df['geo_lat'].between(40, 75)]
df = df[df['geo_lon'].between(15, 185)]

df['rooms'] = df['rooms'].replace(-1, 0)  # в датасете студии обозначены как -1

bad_kitchen = (df['kitchen_area'] < 0) | (df['kitchen_area'] > df['area'] * 0.7)
df.loc[bad_kitchen, 'kitchen_area'] = np.nan
df['kitchen_area'] = df['kitchen_area'].fillna((df['area'] * 0.16).clip(5, 20))

price_per_meter = df['price'] / df['area']
low_border = price_per_meter.quantile(0.005)
high_border = price_per_meter.quantile(0.995)
df = df[price_per_meter.between(low_border, high_border)]

print('было:', before)
print('стало:', len(df))
print('удалили:', before - len(df))

Очистка нужна, чтобы убрать нереальные цены, некорректные этажи, слишком странные площади и выбросы по цене за квадратный метр.

In [ ]:
# Создаем дополнительные признаки для модели
df['floor_ratio'] = df['level'] / df['levels']
df['is_first_floor'] = (df['level'] == 1).astype(int)
df['is_last_floor'] = (df['level'] == df['levels']).astype(int)
df['area_per_room'] = df['area'] / df['rooms'].replace(0, 1)
df['kitchen_ratio'] = df['kitchen_area'] / df['area']
df['log_area'] = np.log1p(df['area'])
df['lat_round'] = df['geo_lat'].round(1).astype(int)
df['lon_round'] = df['geo_lon'].round(1).astype(int)

cat_features = [
    'building_type', 'object_type', 'id_region',
    'postal_code', 'lat_round', 'lon_round', 'month',
]

num_features = [
    'area', 'kitchen_area', 'level', 'levels', 'rooms',
    'geo_lat', 'geo_lon', 'year', 'floor_ratio',
    'is_first_floor', 'is_last_floor', 'area_per_room',
    'kitchen_ratio', 'log_area',
]

features = num_features + cat_features

for col in cat_features:
    df[col] = df[col].fillna(-1).astype(str)

for col in num_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].fillna(df[col].median())

print('признаков:', len(features))
features

Здесь создаются новые признаки: относительный этаж, первый/последний этаж, площадь на комнату и примерная локация.

In [ ]:
# Строим простые графики для теста данных после очистки
plt.figure(figsize=(10, 4))
sns.histplot(df['price'], bins=80)
plt.title('Распределение цены после очистки')
plt.show()

plt.figure(figsize=(10, 4))
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=random_seed), x='area', y='price', alpha=0.3)
plt.title('Связь площади и цены')
plt.show()

Графики нужны для быстрой визуальной теста: после очистки распределение цены и связь с площадью должны выглядеть разумно.

In [ ]:
# Делим данные на обучающую и тестовую части
X = df[features].copy()
y = np.log1p(df['price'])  # логарифмирование стабилизирует разброс цен

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_seed,
)

cat_indexes = [X.columns.get_loc(col) for col in cat_features]

print('train:', X_train.shape)
print('test:', X_test.shape)

Часть данных оставляю для теста, чтобы оценить модель на квартирах, которые она не видела при обучении.

In [ ]:
# Обучаем основную модель CatBoost
model = CatBoostRegressor(
    iterations=1400,
    learning_rate=0.06,
    depth=9,
    l2_leaf_reg=5,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=random_seed,
    early_stopping_rounds=100,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_indexes,
    eval_set=(X_test, y_test),
    use_best_model=True,
)

CatBoost выбран потому, что хорошо работает с табличными данными и умеет обрабатывать категориальные признаки без ручного one-hot encoding.

In [ ]:
# Считаем метрики в рублях, возвращаясь из логарифма к обычной цене
def count_metrics(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'R2': round(r2, 4),
        'MAPE_percent': round(mape, 4),
    }

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_metrics = count_metrics(y_train, train_pred)
test_metrics = count_metrics(y_test, test_pred)

pd.DataFrame([train_metrics, test_metrics], index=['train', 'test'])

Основные метрики качества: `R2`, `MAPE`, `MAE` и `RMSE`. По ним можно понять, насколько сильно прогноз отличается от настоящей цены.

In [ ]:
# Сравниваем фактические цены и прогноз модели
y_real = np.expm1(y_test)
y_pred = np.expm1(test_pred)

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_real, y=y_pred, alpha=0.25)
limit = max(y_real.max(), y_pred.max())
plt.plot([0, limit], [0, limit], color='red')
plt.title('Фактическая цена и прогноз')
plt.xlabel('Фактическая цена')
plt.ylabel('Прогноз')
plt.show()

Если точки расположены близко к красной линии, прогнозы модели близки к реальным значениям.

In [ ]:
# Смотрим, какие признаки сильнее всего влияют на прогноз
importance = pd.DataFrame({
    'feature': features,
    'importance': model.get_feature_importance(),
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance.head(20), x='importance', y='feature')
plt.title('Важность признаков')
plt.show()

importance.head(20)

Feature Importance помогает проверить логику модели: обычно важными оказываются площадь, координаты, регион и тип недвижимости.

In [ ]:
# Обучаем две дополнительные модели для нижней и верхней границы цены
low_alpha = 0.15
high_alpha = 0.85

model_low = CatBoostRegressor(
    iterations=900,
    learning_rate=0.06,
    depth=8,
    l2_leaf_reg=5,
    loss_function=f'Quantile:alpha={low_alpha}',
    eval_metric=f'Quantile:alpha={low_alpha}',
    random_seed=43,
    early_stopping_rounds=80,
    verbose=100,
)

model_high = CatBoostRegressor(
    iterations=900,
    learning_rate=0.06,
    depth=8,
    l2_leaf_reg=5,
    loss_function=f'Quantile:alpha={high_alpha}',
    eval_metric=f'Quantile:alpha={high_alpha}',
    random_seed=44,
    early_stopping_rounds=80,
    verbose=100,
)

model_low.fit(X_train, y_train, cat_features=cat_indexes, eval_set=(X_test, y_test), use_best_model=True)
model_high.fit(X_train, y_train, cat_features=cat_indexes, eval_set=(X_test, y_test), use_best_model=True)

Для бота лучше показывать не только одну цену, но и примерный диапазон. Поэтому отдельно обучаются модели для нижней и верхней границы.

In [ ]:
# Проверяем, как часто реальная цена попадает в рассчитанный интервал
low_pred = np.expm1(model_low.predict(X_test))
high_pred = np.expm1(model_high.predict(X_test))
real_price = np.expm1(y_test)

low_pred, high_pred = np.minimum(low_pred, high_pred), np.maximum(low_pred, high_pred)
interval_coverage = ((real_price >= low_pred) & (real_price <= high_pred)).mean() * 100
avg_interval_width = np.mean(high_pred - low_pred)

print('попадание в интервал %:', round(interval_coverage, 2))
print('средняя ширина интервала:', round(avg_interval_width, 0))

Интервал отражает неопределенность рынка: слишком широкий диапазон будет бесполезным, а слишком узкий часто не будет покрывать реальную цену.

In [ ]:
# Сохраняем обученные модели
model_path = MODELS_DIR / 'input_catboost_model.cbm'
model_low_path = MODELS_DIR / 'input_catboost_low.cbm'
model_high_path = MODELS_DIR / 'input_catboost_high.cbm'

model.save_model(model_path)
model_low.save_model(model_low_path)
model_high.save_model(model_high_path)

print('сохранили основную модель:', model_path)
print('сохранили нижнюю границу:', model_low_path)
print('сохранили верхнюю границу:', model_high_path)

В конце сохраняются три модели: основная модель для цены, модель для нижней границы и модель для верхней границы.

In [ ]:
# Проверяем пример квартиры
example = {
    'area': 60,
    'kitchen_area': 10,
    'level': 7,
    'levels': 16,
    'rooms': 2,
    'geo_lat': 55.75,
    'geo_lon': 37.62,
    'building_type': '2',
    'object_type': '0',
    'id_region': '77',
    'postal_code': '101000',
    'year': 2021,
    'month': '7',
}

one = pd.DataFrame([example])

one['floor_ratio'] = one['level'] / one['levels']
one['is_first_floor'] = (one['level'] == 1).astype(int)
one['is_last_floor'] = (one['level'] == one['levels']).astype(int)
one['area_per_room'] = one['area'] / one['rooms'].replace(0, 1)
one['kitchen_ratio'] = one['kitchen_area'] / one['area']
one['log_area'] = np.log1p(one['area'])
one['lat_round'] = one['geo_lat'].round(1).astype(int).astype(str)
one['lon_round'] = one['geo_lon'].round(1).astype(int).astype(str)

for col in cat_features:
    one[col] = one[col].astype(str)

price = np.expm1(model.predict(one[features])[0])
print('тестовая цена:', round(price))

Если появилась тестовая цена, значит модель может сделать прогноз для одного примера квартиры.